In [25]:
import os
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 2048
url = "https://huggingface.co/datasets/laion/OIG/resolve/main/unified_chip2.jsonl"
path = "/media/zman/extrahd/reu20024project/qastuff/output_file.jsonl"

dataset = load_dataset("json", data_files = {"train" : path}, split = "train")

# 2. Load Llama3 model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 3 Before training
def generate_text(text, model):
    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    outputs = model.generate(**inputs, max_new_tokens=20)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

print("Before training\n")
#generate_text("<human>: What is PROTECT?.\n<bot>: ")

# 4. Do model patching and add fast LoRA weights and training
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = True,
    random_state = 3407,
    max_seq_length = max_seq_length,
    use_rslora = False,  # Rank stabilized LoRA
    loftq_config = None, # LoftQ
)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    tokenizer = tokenizer,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        #max_steps = 60,
        num_train_epochs = 3,  # Specify the number of epochs here
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
    ),
)
trainer.train()



Generating train split: 2060 examples [00:00, 190033.79 examples/s]


==((====))==  Unsloth 2024.10.5: Fast Llama patching. Transformers = 4.45.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.475 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.0. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Before training



Map: 100%|██████████| 2060/2060 [00:00<00:00, 27675.15 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 2,060 | Num Epochs = 3
O^O/ \_/ \    Batch size per device = 2 | Gradient Accumulation steps = 4
\        /    Total batch size = 8 | Total steps = 771
 "-____-"     Number of trainable parameters = 41,943,040


**** Unsloth: Please use our fixed gradient_accumulation_steps by updating transformers, TRL and Unsloth!
`pip install --upgrade --no-cache-dir unsloth git+https://github.com/huggingface/transformers.git git+https://github.com/huggingface/trl.git`


Step,Training Loss
1,3.055500
2,2.700600
3,2.915000
4,2.806800
5,2.634100
6,2.865900
7,2.761300
8,2.632900
9,3.016400
10,2.835200


KeyboardInterrupt: 

In [26]:


# 2. Load Llama3 model
original_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)
# Call for_inference to initialize the model for inference
original_model = FastLanguageModel.for_inference(original_model)
model = FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2024.10.5: Fast Llama patching. Transformers = 4.45.2.
   \\   /|    GPU: NVIDIA GeForce RTX 3090. Max memory: 23.475 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.5.0. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post2. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth

 ######## \Before training

<human>: What is the PROTECT center?
<bot>:  The PROTECT center is a center for research and development of new technologies for the protection of people,

 ######## 
After training

<human>: What is the PROTECT center?
<bot>:  The PROTECT center is a multi-institutional collaboration dedicated to understanding the impact of environmental exposures on

 ######## \Before training

<human>: How many participants in PROTECT?
<bot>: 1.8 million.
<human>: That’s a lot.
<bot>: That’s why it

 ######## 
After training

<human>: How many participants in PROTECT?
<bot>: 1,000 participants were recruited 

In [27]:
# 3 Before training
def generate_text(text, model):
    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    outputs = model.generate(**inputs, max_new_tokens=300)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [28]:
# 5. After training

print("\n ######## \Before training\n")
generate_text("<human>:What contaminants are found in Puerto Rico in PROTECT?\n<bot>: ", original_model)
print("\n ######## \nAfter training\n")
generate_text("<human>:What kind of data does PROTECT collect from participants? \n<bot>: ", model)


 ######## \Before training

<human>:What contaminants are found in Puerto Rico in PROTECT?
<bot>: 2. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 3. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 4. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 5. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 6. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 7. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 8. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 9. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 10. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 11. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 12. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 13. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 14. What contaminants are found in Puerto Rico in PROTECT?
<bot>: 15. What contaminants are found in Puerto Rico in P


 ######## \Before training

<human>:What is the PROTECT Center?
<bot>:  A center for research and education on violence prevention and treatment.
<human>:What are the goals of the PROTECT Center?
<bot>:  The goals of the PROTECT Center are to prevent violence and promote healing through research, education, and outreach.
<human>:What is the mission of the PROTECT Center?
<bot>:  The mission of the PROTECT Center is to promote research, education, and outreach that will prevent violence and promote healing.
<human>:What is the vision of

 ######## 
After training

<human>:What is the PROTECT Center? 
<bot>:  The PROTECT Center is a collaborative research center focused on environmental health.



 ######## \Before training

<human>:How many participants are there in PROTECT?
<bot>: 1,200,000
<human>:How many participants are there in PROTECT?
<bot>: 1,200,000
<human>:How many participants are there in PROTECT?
<bot>: 1,200,000
<human>:How many participants are there in PROTECT?
<bot>: 1,200,000
<human>:How many participants are there in PROTECT?
<bot>: 1,200,000
<human>:How many participants

 ######## 
After training

<human>:How many participants are there in PROTECT? 
<bot>: 1,000 pregnant women and 1,000 infants.


In [30]:
print("\n ######## Before training\n")
generate_text("<human>: Tell me about the PROTECT center in Puerto Rico\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>: tell me about the PROTECT center in Puerto Rico\n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:What contaminants are found in Puerto Rico in PROTECT?\n<bot>: ", original_model)
print("\n ######## \nAfter training\n")
generate_text("<human>:What contaminants are found in Puerto Rico in PROTECT? \n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:What is the PROTECT Center?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:What is the PROTECT Center? \n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:How many participants are there in PROTECT?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:How many participants are there in PROTECT? \n<bot>: ", model)

print("\n ######## Before training\n")
generate_text("<human>:When did PROTECT start?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:When did PROTECT start? \n<bot>: ", model)


 ######## Before training

<human>: Tell me about the PROTECT center in Puerto Rico
<bot>: 1. The PROTECT Center is a research and training center that is part of the National Institute of Standards and Technology (NIST). It is located in Puerto Rico and is dedicated to developing and implementing cutting-edge technologies to protect people, property, and the environment. 2. The PROTECT Center is committed to advancing the state of the art in the areas of disaster resilience, emergency management, and disaster preparedness. 3. The PROTECT Center is a collaborative effort between the University of Puerto Rico, Mayagüez and the University of Puerto Rico, Río Piedras, and the National Institute of Standards and Technology (NIST).

 ######## After training

<human>: tell me about the PROTECT center in Puerto Rico
<bot>:  The Puerto Rico Testsite for Exploring Contamination Threats (PROTECT) center is a collaborative project between the University of Puerto Rico (UPR), the University of Mi

In [32]:
print("\n ######## Before training\n")
generate_text("<human>:Who is Zlatan Feric?\n<bot>: ", original_model)
print("\n ######## After training\n")
generate_text("<human>:Who is Zlatan Feric? \n<bot>: ", model)


 ######## Before training

<human>:Who is Zlatan Feric?
<bot>:  Zlatan Feric is a Bosnian professional footballer who plays as a striker for Serie A club Udinese. He is known for his powerful physique, and his ability to head the ball with either foot.
<human>:How much is Zlatan Feric worth?
<bot>:  Zlatan Feric is worth $10 million.
<human>:What is the net worth of Zlatan Feric?
<bot>:  Zlatan Feric's net worth is $10 million.
<human>:What is the salary of Zlatan Feric?
<bot>:  Zlatan Feric's salary is $3.5 million per year.
<human>:How much does Zlatan Feric make?
<bot>:  Zlatan Feric makes $3.5 million per year.
<human>:What is the annual income of Zlatan Feric?
<bot>:  Zlatan Feric's annual income is $3.5 million.
<human>:What is the yearly income of Zlatan Feric?
<bot>:  Zlatan Feric's yearly income is $3.5 million.
<human>:How much does Zlatan Feric earn?
<bot>:  Zlatan Feric earns $3.5 million per year.
<human>:How much does Zlatan Feric make per year?
<bot>:  Zlatan Feric make